## load data in Document objects with lang chain 

In [23]:
from langchain_community.document_loaders import TextLoader

In [24]:

loader = TextLoader("data/txt/python.txt", encoding="utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'data/txt/python.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python emphasizes clean syntax and ease of use, making it an excellent choice for beginners and experienced developers alike. Its design philosophy encourages writing clear and concise code, which reduces the complexity often associated with programming.\n\nPython is widely used in various fields such as web development, data science, artificial intelligence, machine learning, automation, and scientific computing. Popular frameworks like Django and Flask support web development, while libraries such as NumPy, pandas, and TensorFlow make Python a powerful tool for data analysis and machine learning.\n\nOne of Python’s key strengths is its large and active community, which contributes to a vast ecosystem of libraries and tools. This allows developers to quickly build a

## directory loader

In [27]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "data/txt",
    glob="**/*.txt", ##pattern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)

directory_documnets = dir_loader.load()
directory_documnets

100%|██████████| 2/2 [00:00<00:00, 89.36it/s]


[Document(metadata={'source': 'data\\txt\\ml.txt'}, page_content='Machine Learning (ML) is a branch of Artificial Intelligence (AI) that focuses on enabling computers to learn from data and make decisions or predictions without being explicitly programmed for every task. Instead of following fixed instructions, ML systems identify patterns in data and improve their performance over time through experience.\n\nAt its core, machine learning uses algorithms to process large amounts of data, detect meaningful relationships, and generate models that can be used for tasks such as classification, prediction, and decision-making. These models are trained using historical data and can adapt as new data becomes available.\n\nThere are three main types of machine learning:\n\n1. Supervised Learning – In this approach, the model is trained on labeled data, where the correct output is already known. It is commonly used for tasks like spam detection and price prediction.\n\n2. Unsupervised Learning 

## pdf loader

In [64]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "data/pdf",
    glob="**/*.pdf", ##pattern to match files
    loader_cls=PyMuPDFLoader, ##loader class to use
)

directory_documnets = dir_loader.load()
directory_documnets

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-26T12:03:52+00:00', 'source': 'data\\pdf\\ml_intro.pdf', 'file_path': 'data\\pdf\\ml_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-04-26T12:03:52+00:00', 'trapped': '', 'modDate': "D:20260426120352+00'00'", 'creationDate': "D:20260426120352+00'00'", 'page': 0}, page_content='Machine Learning (ML) is a subset of Artificial Intelligence (AI) that allows systems to learn from\ndata\nand improve their performance without being explicitly programmed.\nML algorithms identify patterns in data and use them to make predictions or decisions.\nIt is widely used in applications like recommendation systems, image recognition, and fraud\ndetection.\nTypes of Machine Learning:\n- Supervised Learning\n- Unsupervised Learning\n- Reinforcement Learning\nMachine L

In [66]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

def split_documnets(documents,chunk_size=1000,chunk_overlap=200):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n","\n"," ",""]
        )
    
    split_documnets = text_spliter.split_documents(documents)

    return split_documnets

In [67]:
chunks = split_documnets(directory_documnets)

In [37]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-26T12:03:52+00:00', 'source': 'data\\pdf\\ml_intro.pdf', 'file_path': 'data\\pdf\\ml_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-04-26T12:03:52+00:00', 'trapped': '', 'modDate': "D:20260426120352+00'00'", 'creationDate': "D:20260426120352+00'00'", 'page': 0}, page_content='Machine Learning (ML) is a subset of Artificial Intelligence (AI) that allows systems to learn from\ndata\nand improve their performance without being explicitly programmed.\nML algorithms identify patterns in data and use them to make predictions or decisions.\nIt is widely used in applications like recommendation systems, image recognition, and fraud\ndetection.\nTypes of Machine Learning:\n- Supervised Learning\n- Unsupervised Learning\n- Reinforcement Learning\nMachine L

## EmBeddings And VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple, Any
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
import os

In [8]:
class EmbeddingManager:
    def __init__(self, model_name:str = "all-MiniLM-l6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loadded successfully.")
            print(f"Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading Model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of text

        Args: 
            texts: List of text strings to embed

        returns: 
            numpy array of mebeddings with shape (len(texts), embedding_dim) 
        """
        if not self.model:
            raise ValueError('Model not loaded')
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self)-> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError('Model not loaded')
        return self.model.get_embedding_dimension()
    
## initialize the embedding manager

embeding_manager = EmbeddingManager()



loading embedding model : all-MiniLM-l6-v2


e:\Python\RAG\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aryan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-l6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7633.15it/s]


Model loadded successfully.
Embedding dimension: 384


## vector store

In [53]:
class VectoreStore:

    def __init__(self, collection_name: str = "txt_documents", persist_directory: str ="data/vector_store"):
        """ 
            Initialize the vectore store
            Args: 
                collection_name: name of the chromaDb collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize croma db client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"discription": "txt document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documnets in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise
    
    def add_documnets(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and its embeddings to store"""

        if len(embeddings) != len(documents):
            raise ValueError('Number of documents and embeddings do not match')

        print(f"Adding {len(documents)} documents to vector store")

        #prepare data for ChromaDb

        ids             = []
        metadatas       = []
        documents_text   = []
        embeddings_list = []

        for  i, (doc,embedding) in  enumerate(zip(documents,embeddings)):
            #generate unique Id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadta
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try: 
            self.collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documnets to vector store")
            print(f"total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documnets ito vector store: {e}")

vectorstore = VectoreStore()
vectorstore
    
        

Vector store initialized. Collection: txt_documents
Existing documnets in collection: 4


In [39]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-26T12:03:52+00:00', 'source': 'data\\pdf\\ml_intro.pdf', 'file_path': 'data\\pdf\\ml_intro.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-04-26T12:03:52+00:00', 'trapped': '', 'modDate': "D:20260426120352+00'00'", 'creationDate': "D:20260426120352+00'00'", 'page': 0}, page_content='Machine Learning (ML) is a subset of Artificial Intelligence (AI) that allows systems to learn from\ndata\nand improve their performance without being explicitly programmed.\nML algorithms identify patterns in data and use them to make predictions or decisions.\nIt is widely used in applications like recommendation systems, image recognition, and fraud\ndetection.\nTypes of Machine Learning:\n- Supervised Learning\n- Unsupervised Learning\n- Reinforcement Learning\nMachine L

## convert text to embeddings

In [68]:
texts = [doc.page_content for doc in chunks]
texts

['Machine Learning (ML) is a subset of Artificial Intelligence (AI) that allows systems to learn from\ndata\nand improve their performance without being explicitly programmed.\nML algorithms identify patterns in data and use them to make predictions or decisions.\nIt is widely used in applications like recommendation systems, image recognition, and fraud\ndetection.\nTypes of Machine Learning:\n- Supervised Learning\n- Unsupervised Learning\n- Reinforcement Learning\nMachine Learning is transforming industries and is a key driver of innovation in modern technology.',
 'PHP (Hypertext Preprocessor) is a widely-used open-source scripting language designed primarily\nfor web development.\nIt is especially suited for creating dynamic and interactive web pages and can be embedded directly\ninto HTML.\nPHP runs on the server side, meaning the code is executed on the server and the result is sent to\nthe client’s browser.\nIt is commonly used with databases like MySQL to build data-driven app

In [69]:
## generate the embeddings

embeddings = embeding_manager.generate_embeddings(texts)

## store in vector database
vectorstore.add_documnets(chunks, embeddings)

Generating embeddings for 3 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.77it/s]

Generated embeddings with shape : (3, 384)
Adding 3 documents to vector store
Successfully added 3 documnets to vector store
total documents in collection : 7


## Retriver Pipeline From Vector Store

In [56]:
class RagRetriver:
    def __init__(self,vectore_store:VectoreStore,embeding_manager: EmbeddingManager):
        self.vector_store = vectore_store
        self.embedding_manager = embeding_manager

    def retrieve(self, query:str, top_k: int = 5, score_threshold : float = 0.0) -> List[Dict[str,Any]]:
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )

            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids       = results['ids'][0]

                for i,(doc_id, document, metadata, distance) in  enumerate(zip(ids, documents,metadatas,distances)):
                    #converts distance  to similarity scores
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance': distance,
                            'rank':i+1
                        })
                    print(f"Retrived {len(retrived_docs)} documents (after filtering)")
            else :
                print("No documnets found")

            return retrived_docs

        except Exception as e:
            print(f'error during retrival{e}')
            return []
        
rag_retriver = RagRetriver(vectorstore,embeding_manager)

In [57]:
rag_retriver

In [70]:
rag_retriver.retrieve("what is php")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.30it/s]

Generated embeddings with shape : (1, 384)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)


[{'id': 'doc_3dbc1485_1',
  'content': 'PHP (Hypertext Preprocessor) is a widely-used open-source scripting language designed primarily\nfor web development.\nIt is especially suited for creating dynamic and interactive web pages and can be embedded directly\ninto HTML.\nPHP runs on the server side, meaning the code is executed on the server and the result is sent to\nthe client’s browser.\nIt is commonly used with databases like MySQL to build data-driven applications such as websites,\ncontent management systems, and e-commerce platforms.\nKey features of PHP include:\n- Easy integration with HTML\n- Strong database support\n- Cross-platform compatibility\n- Large community and extensive documentation\nPHP powers many popular platforms and remains a core technology in web development.\nIn conclusion, PHP is a practical and efficient language for building dynamic web applications and\ncontinues to play a vital role in the web ecosystem.',
  'metadata': {'modDate': "D:20260426125535+00